# Aquaculture Model Inference with Feature Selection

This notebook demonstrates how to use a trained model for inference on the competition test dataset, 
including how to work with feature selection capabilities.

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os
import sys
import json
import pickle
import yaml
from pathlib import Path

# Add the parent directory to the system path to import local modules
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# Import our custom modules
from src.trainer import Trainer
from aquaculture.feature_selection import FeatureSelector
from aquaculture.feature_engineering import AquacultureFeatureEngineer

# For reproducibility
np.random.seed(42)

In [ ]:
# Define paths
DATA_DIR = Path("../data")
EXPERIMENTS_DIR = Path("../experiments")

# Verify directories exist
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR.resolve()}")

if not EXPERIMENTS_DIR.is_dir():
    raise FileNotFoundError(f"Experiments directory not found: {EXPERIMENTS_DIR.resolve()}")


In [ ]:
# Find the latest experiment directory
experiment_dirs = [d for d in EXPERIMENTS_DIR.iterdir() if d.is_dir()]
if not experiment_dirs:
    raise FileNotFoundError("No experiment directories found in ../experiments")

latest_experiment = max(experiment_dirs, key=lambda d: d.stat().st_mtime)
print(f"Using latest experiment: {latest_experiment}")


In [ ]:
# Load the trained model and related artifacts
print("Loading trained model and artifacts...")
# Try to load the trainer first (which includes feature selector)
trainer_path = latest_experiment / "trainer.pkl"
if trainer_path.exists():
    with open(trainer_path, 'rb') as f:
        trainer = pickle.load(f)
    print("Trainer loaded successfully (includes feature selector)")
else:
    # Fallback: load components individually if trainer.pkl is not available
    print("Trainer.pkl not found, loading components individually...")
    
    # Load the actual model
    model_path = latest_experiment / "models" / "best_model.pkl"
    if model_path.exists():
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        print(f"Model type: {type(model).__name__}")
    else:
        raise FileNotFoundError(f"Model file not found: {model_path}")

    # Create a minimal trainer-like object for compatibility
    trainer = type('TrainerStub', (), {})()
    trainer.model = model

    # Add predict and predict_proba methods that delegate to the model
    trainer.predict = lambda X, training=False: trainer.model.predict(X)
    trainer.predict_proba = lambda X, training=False: trainer.model.predict_proba(X)

    # Try to load feature names if available
    feature_names_path = latest_experiment / "features" / "feature_names.json"
    if feature_names_path.exists():
        with open(feature_names_path, 'r') as f:
            trainer.feature_names = json.load(f)
        print(f"Loaded {len(trainer.feature_names)} feature names")
    else:
        # Fallback: try to get feature names from the model if possible
        trainer.feature_names = None
        print("Warning: Feature names file not found, will use all test features")

    # Load config to get model_type and other settings
    config_path = latest_experiment / "config.yaml"
    feature_engineering_config = None  # Initialize to None
    if config_path.exists():
        try:
            # Use yaml.load instead of safe_load to handle Python-specific types like tuples
            # Note: This is safe because we trust our own config files
            with open(config_path, 'r') as f:
                config_data = yaml.load(f, Loader=yaml.FullLoader)
           
            # Extract feature engineering configuration
            if 'feature_engineering_config' in config_data:
                feature_engineering_config = config_data['feature_engineering_config']
                print("Loaded feature engineering configuration from training config")
            else:
                print("Warning: No feature_engineering_config found in config file")
        except Exception as e:
            print(f"Warning: Could not load config file: {e}")
            # feature_engineering_config remains None
    else:
        print("Warning: Config file not found")

    # Create a simple config-like object for trainer.config
    trainer.config = type('ConfigStub', (), {})()
    trainer.config.model_type = config_data.get('model_type', 'unknown') if 'config_data' in locals() and config_data is not None else 'unknown'

    print(f"Model type: {trainer.config.model_type}")

In [ ]:
# Load test data
print("Loading test data...")
test_df = pd.read_csv(DATA_DIR / 'Test.csv')
print(f"Test data shape: {test_df.shape}")


In [ ]:
# Load sample submission to get format
sample_submission = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')
print(f"Sample submission shape: {sample_submission.shape}")


In [ ]:
# Prepare test features (same as used during training)
print("Preparing test features using the same process as training...")
# Load the raw test data (same format as training data)
test_df = pd.read_csv(DATA_DIR / 'Test.csv')
print(f"Raw test data shape: {test_df.shape}")

# The target column is 'label' in the training data
# Feature columns are all columns except ID and label
test_feature_cols = [col for col in test_df.columns if col not in ['ID']]
X_test_raw = test_df[test_feature_cols].values
n_samples_test = X_test_raw.shape[0]
X_test = X_test_raw.reshape(n_samples_test, 12, 12)
print(f"Reshaped test data shape: {X_test.shape}")

# Save raw data for debugging
raw_data_path = latest_experiment / "debug_raw_test_data.csv"
raw_data_df = pd.DataFrame(X_test_raw)
raw_data_df.to_csv(raw_data_path, index=False)
print(f"✓ Saved raw test data to: {raw_data_path}")

In [ ]:
# ALWAYS use the trainer's pre-fitted components if available
if trainer is not None and hasattr(trainer, 'feature_engineer') and trainer.feature_engineer is not None:
    print("✓ Using pre-fitted feature engineering components from trainer")
    feature_engineer = trainer.feature_engineer
    feature_selector = getattr(trainer, 'feature_selector', None)
    
    # Debug information
    print(f"  Feature engineer type: {type(feature_engineer)}")
    print(f"  Feature selector: {feature_selector is not None}")
    if feature_selector is not None:
        print(f"  Selection method: {getattr(feature_selector, 'selection_method', 'None')}")
        print(f"  Selection kwargs: {getattr(feature_selector, 'selection_kwargs', {})}")
    
    # Save features after engineering but before selection for debugging
    if feature_selector is not None:
        # When selector is available, get engineered features before selection
        X_test_engineered = feature_engineer.transform(X_test, training=False)
        engineered_data_path = latest_experiment / "debug_engineered_features_before_selection.csv"
        engineered_df = X_test_engineered
        engineered_df.to_csv(engineered_data_path, index=False)
        print(f"✓ Saved engineered features (before selection) to: {engineered_data_path}")
        print(f"  Engineered features shape: {X_test_engineered.shape}")
    
    # Use the trainer's pipeline (feature engineering + selection if available)
    print("Transforming test data using trainer's pipeline...")
    try:
        if feature_selector is not None:
            # Apply both feature engineering and feature selection (EXACTLY like during training)
            X_test_features = feature_selector.transform(X_test, training=False)
            print(f"✓ Transformed test data with feature selection: {X_test_features.shape[1]} features")
            
            # Save final features after selection for debugging
            selected_data_path = latest_experiment / "debug_selected_features_after_selection.csv"
            selected_df = X_test_features
            selected_df.to_csv(selected_data_path, index=False)
            print(f"✓ Selected features (after selection) to: {selected_data_path}")
            print(f"  Selected features shape: {X_test_features.shape}")
        else:
            # Only feature engineering (if selection was disabled during training)
            X_test_features = feature_engineer.transform(X_test, training=False)
            print(f"✓ Transformed test data (feature engineering only): {X_test_features.shape[1]} features")
            
            # Save final features after engineering (no selection) for debugging
            engineered_data_path = latest_experiment / "debug_engineered_features_no_selection.csv"
            engineered_df = X_test_features
            engineered_df.to_csv(engineered_data_path, index=False)
            print(f"✓ Engineered features (no selection) to: {engineered_data_path}")
            print(f"  Engineered features shape: {X_test_features.shape}")
    except Exception as e:
        print(f"⚠ Error transforming with trainer pipeline: {e}")
        print("Falling back to manual reconstruction...")
        # Fallback: manually recreate what the trainer would do
        
        # Recreate feature engineer with same settings
        fec = None
        config_path = latest_experiment / "config.yaml"
        if config_path.exists():
            with open(config_path, 'r') as f:
                config_data = yaml.load(f, Loader=yaml.FullLoader)
                fec = config_data.get('feature_engineering_config', {})
        
        if fec is not None:
            manual_fe = AquacultureFeatureEngineer(
                simulate_mask=False,
                random_state=fec.get('random_state', 42),
                window_length_probs=tuple(fec.get('window_length_probs', (1/3, 1/3, 1/3))),
                start_month_distribution=fec.get('start_month_distribution'),
                s2_monthly_dropout=fec.get('s2_monthly_dropout', [0.0]*12),
                include_optical=fec.get('include_optical', True),
                include_sar=fec.get('include_sar', True),
                include_cross_sensor_features=fec.get('include_cross_sensor_features', True),
                include_temporal_statistics=fec.get('include_temporal_statistics', True),
                include_directional_vote=fec.get('include_directional_vote', True),
                include_metadata=fec.get('include_metadata', True)
            )
        else:
            manual_fe = AquacultureFeatureEngineer(simulate_mask=False)
        
        manual_fe.fit(X_test)  # Fit on test data to establish feature names
        
        # Apply feature selection if configured
        manual_fs = None
        if (hasattr(trainer, 'feature_selection_enabled') and 
            getattr(trainer, 'feature_selection_enabled', False) and
            hasattr(trainer, 'feature_selection_method') and
            hasattr(trainer, 'feature_selection_kwargs')):
            try:
                manual_fs = FeatureSelector(
                    manual_fe,
                    selection_method=getattr(trainer, 'feature_selection_method', 'groups'),
                    **getattr(trainer, 'feature_selection_kwargs', {})
                )
                print(f"✓ Created manual feature selector: {getattr(trainer, 'feature_selection_method')}")
            except Exception as e2:
                print(f"⚠ Could not create manual feature selector: {e2}")
                manual_fs = None
        
        if manual_fs is not None:
            X_test_features = manual_fs.transform(X_test, training=False)
            print(f"✓ Manual transform with selection: {X_test_features.shape[1]} features")
            
            # Save final features after manual selection for debugging
            selected_data_path = latest_experiment / "debug_manual_selected_features.csv"
            selected_df = X_test_features
            selected_df.to_csv(selected_data_path, index=False)
            print(f"✓ Manual selected features to: {selected_data_path}")
            print(f"  Manual selected features shape: {X_test_features.shape}")
        else:
            X_test_features = manual_fe.transform(X_test, training=False)
            print(f"✓ Manual transform (no selection): {X_test_features.shape[1]} features")
            
            # Save final features after manual engineering (no selection) for debugging
            engineered_data_path = latest_experiment / "debug_manual_engineered_features.csv"
            engineered_df = X_test_features
            engineered_df.to_csv(engineered_data_path, index=False)
            print(f"✓ Manual engineered features to: {engineered_data_path}")
            print(f"  Manual engineered features shape: {X_test_features.shape}")
else:
    print("⚠ Trainer components not available, creating feature engineer from config")
    # Fallback: create feature engineer from configuration
    config_path = latest_experiment / "config.yaml"
    if config_path.exists():
        try:
            with open(config_path, 'r') as f:
                config_data = yaml.load(f, Loader=yaml.FullLoader)
            fec = config_data.get('feature_engineering_config', {})
            from aquaculture.feature_engineering import AquacultureFeatureEngineer
            feature_engineer = AquacultureFeatureEngineer(
                simulate_mask=False,
                random_state=fec.get('random_state', 42),
                window_length_probs=tuple(fec.get('window_length_probs', (1/3, 1/3, 1/3))),
                start_month_distribution=fec.get('start_month_distribution'),
                s2_monthly_dropout=fec.get('s2_monthly_dropout', [0.0]*12),
                include_optical=fec.get('include_optical', True),
                include_sar=fec.get('include_sar', True),
                include_cross_sensor_features=fec.get('include_cross_sensor_features', True),
                include_temporal_statistics=fec.get('include_temporal_statistics', True),
                include_directional_vote=fec.get('include_directional_vote', True),
                include_metadata=fec.get('include_metadata', True)
            )
            print("✓ Created feature engineer from config")
            
            # Try to create feature selector from config if available
            feature_selector = None
            if fec.get('feature_selection_enabled', False):
                from aquaculture.feature_selection import FeatureSelector
                try:
                    feature_selector = FeatureSelector(
                        feature_engineer,
                        selection_method=fec.get('feature_selection_method', 'groups'),
                        **fec.get('feature_selection_kwargs', {})
                    )
                    print(f"✓ Created feature selector from config: {fec.get('feature_selection_method')}")
                except Exception as e:
                    print(f"⚠ Could not create feature selector from config: {e}")
                    feature_selector = None
            
            if feature_selector is not None:
                X_test_features = feature_selector.transform(X_test, training=False)
                print(f"✓ Transformed with config-based selection: {X_test_features.shape[1]} features")
                
                # Save final features after config-based selection for debugging
                selected_data_path = latest_experiment / "debug_config_selected_features.csv"
                selected_df = X_test_features
                selected_df.to_csv(selected_data_path, index=False)
                print(f"✓ Config-selected features to: {selected_data_path}")
                print(f"  Config-selected features shape: {X_test_features.shape}")
            else:
                X_test_features = feature_engineer.transform(X_test, training=False)
                print(f"✓ Transformed with config (no selection): {X_test_features.shape[1]} features")
                
                # Save final features after config-based engineering (no selection) for debugging
                engineered_data_path = latest_experiment / "debug_config_engineered_features.csv"
                engineered_df = X_test_features
                engineered_df.to_csv(engineered_data_path, index=False)
                print(f"✓ Config-engineered features to: {engineered_data_path}")
                print(f"  Config-engineered features shape: {X_test_features.shape}")
        except Exception as e:
            print(f"⚠ Error creating feature engineer from config: {e}")
            # Last resort fallback
            from aquaculture.feature_engineering import AquacultureFeatureEngineer
            feature_engineer = AquacultureFeatureEngineer(simulate_mask=False)
            X_test_features = feature_engineer.transform(X_test, training=False)
            print(f"✓ Transformed test data (last resort): {X_test_features.shape[1]} features")
            
            # Save final features after last resort engineering (no selection) for debugging
            engineered_data_path = latest_experiment / "debug_last_resort_engineered_features.csv"
            engineered_df = X_test_features
            engineered_df.to_csv(engineered_data_path, index=False)
            print(f"✓ Last resort engineered features to: {engineered_data_path}")
            print(f"  Last resort engineered features shape: {X_test_features.shape}")
    else:
        print("⚠ Config file not found, using default feature engineer")
        from aquaculture.feature_engineering import AquacultureFeatureEngineer
        feature_engineer = AquacultureFeatureEngineer(simulate_mask=False)
        X_test_features = feature_engineer.transform(X_test, training=False)
        print(f"✓ Transformed test data (default): {X_test_features.shape[1]} features")
        
        # Save final features after default engineering (no selection) for debugging
        engineered_data_path = latest_experiment / "debug_default_engineered_features.csv"
        engineered_df = X_test_features
        engineered_df.to_csv(engineered_data_path, index=False)
        print(f"✓ Default engineered features to: {engineered_data_path}")
        print(f"  Default engineered features shape: {X_test_features.shape}")

# Get the feature names for reference - load directly from training when available
feature_names_path = latest_experiment / "features" / "feature_names.json"
if feature_names_path.exists():
    try:
        with open(feature_names_path, 'r') as f:
            test_feature_names = json.load(f)
        print(f"✓ Loaded {len(test_feature_names)} feature names from training")
    except Exception as e:
        print(f"⚠ Error loading feature names from JSON: {e}")
        print("Falling back to extraction from transformed data...")
        # Fallback to original method
        if hasattr(X_test_features, 'columns'):
            test_feature_names = list(X_test_features.columns)
        elif hasattr(feature_selector, 'get_feature_names_out') and feature_selector is not None:
            # Try to get feature names from the selector (most accurate)
            try:
                test_feature_names = list(feature_selector.get_feature_names_out())
            except:
                test_feature_names = [f"feature_{i}" for i in range(X_test_features.shape[1])]
        elif hasattr(feature_engineer, 'get_feature_names_out'):
            # Try to get feature names from the engineer
            test_feature_names = list(feature_engineer.get_feature_names_out())
        else:
            test_feature_names = [f"feature_{i}" for i in range(X_test_features.shape[1])]
else:
    print("⚠ Feature names file not found, extracting from transformed data...")
    # Fallback to original method
    if hasattr(X_test_features, 'columns'):
        test_feature_names = list(X_test_features.columns)
    elif hasattr(feature_selector, 'get_feature_names_out') and feature_selector is not None:
        # Try to get feature names from the selector (most accurate)
        try:
            test_feature_names = list(feature_selector.get_feature_names_out())
        except:
            test_feature_names = [f"feature_{i}" for i in range(X_test_features.shape[1])]
    elif hasattr(feature_engineer, 'get_feature_names_out'):
        # Try to get feature names from the engineer
        test_feature_names = list(feature_engineer.get_feature_names_out())
    else:
        test_feature_names = [f"feature_{i}" for i in range(X_test_features.shape[1])]

print(f"Generated {len(test_feature_names)} features for test data")
if len(test_feature_names) > 0:
    print(f"First 5 feature names: {list(test_feature_names)[:5]}")
    print(f"Last 5 feature names: {list(test_feature_names)[-5:]}")

# Validate feature names match those used during training
if hasattr(trainer, 'feature_names') and trainer.feature_names is not None:
    train_features = set(trainer.feature_names)
    infer_features = set(test_feature_names)
    
    if train_features == infer_features:
        print(f"✓ Feature names match between training and inference ({len(train_features)} features)")
    else:
        print("⚠ WARNING: Feature names mismatch between training and inference!")
        print(f"  Training features: {len(train_features)}")
        print(f"  Inference features: {len(infer_features)}")
        
        # Show specific differences
        train_only = train_features - infer_features
        infer_only = infer_features - train_features
        
        if train_only:
            print(f"  Features in training but not inference: {sorted(train_only)}")
        if infer_only:
            print(f"  Features in inference but not training: {sorted(infer_only)}")

# For prediction, we'll use the feature values
X_test_for_prediction = X_test_features.values if hasattr(X_test_features, 'values') else X_test_features
print(f"Test features array shape: {X_test_for_prediction.shape}")

# Add verification that features are consistent (check for NaN or inf values)
if hasattr(X_test_for_prediction, 'shape'):
    nan_count = np.isnan(X_test_for_prediction).sum()
    inf_count = np.isinf(X_test_for_prediction).sum()
    print(f"✓ Feature verification: {nan_count} NaN values, {inf_count} Inf values in features")
    
    # Save feature statistics for debugging
    feature_stats_path = latest_experiment / "debug_feature_statistics.csv"
    if hasattr(X_test_for_prediction, 'shape') and len(X_test_for_prediction.shape) == 2:
        # Calculate basic statistics for each feature
        feature_means = np.nanmean(X_test_for_prediction, axis=0)
        feature_stds = np.nanstd(X_test_for_prediction, axis=0)
        feature_mins = np.nanmin(X_test_for_prediction, axis=0)
        feature_maxs = np.nanmax(X_test_for_prediction, axis=0)
        
        stats_df = pd.DataFrame({
            'feature_index': range(len(feature_means)),
            'mean': feature_means,
            'std': feature_stds,
            'min': feature_mins,
            'max': feature_maxs
        })
        stats_df.to_csv(feature_stats_path, index=False)
        print(f"✓ Feature statistics saved to: {feature_stats_path}")

In [ ]:
# Make predictions using the engineered features
print("Making predictions...")
# Get raw predictions and probabilities for debugging
raw_predictions = trainer.model.predict(X_test_for_prediction)
print(f"Raw predictions shape: {raw_predictions.shape}")
print(f"Raw predictions type: {type(raw_predictions)}")
print(f"Raw predictions first 10: {raw_predictions[:10]}")
print(f"Raw predictions unique values: {np.unique(raw_predictions)}")
print(f"Raw predictions bincount: {np.bincount(raw_predictions)}")

# Also get probabilities to check distribution
try:
    raw_probabilities = trainer.model.predict_proba(X_test_for_prediction)[:, 1]  # Probability of positive class
    print(f"Raw probabilities shape: {raw_probabilities.shape}")
    print(f"Raw probabilities first 10: {raw_probabilities[:10]}")
    print(f"Raw probabilities min/max/mean: {raw_probabilities.min():.4f} / {raw_probabilities.max():.4f} / {raw_probabilities.mean():.4f}")
    print(f"Raw probabilities std: {raw_probabilities.std():.4f}")
    
    # Check what threshold would give us
    threshold_05_count = np.sum(raw_probabilities >= 0.5)
    print(f"Predictions >= 0.5 threshold: {threshold_05_count} out of {len(raw_probabilities)}")
    print(f"Predictions < 0.5 threshold: {len(raw_probabilities) - threshold_05_count} out of {len(raw_probabilities)}")
    
except AttributeError as e:
    print(f"Could not get probabilities: {e}")
    raw_probabilities = None

# Use the predictions for submission
predictions = raw_predictions
print(f"Final predictions shape: {predictions.shape}")
print(f"Final predictions first 5: {predictions[:5]}")

# Get probabilities for ROC AUC calculation (if available)
try:
    probabilities = trainer.model.predict_proba(X_test_for_prediction)[:, 1]  # Probability of positive class
    has_proba = True
    print(f"Probabilities for submission shape: {probabilities.shape}")
except AttributeError:
    # If predict_proba is not available, use predictions as probabilities
    probabilities = predictions.astype(float)
    has_proba = False
    print("Warning: predict_proba not available, using predictions as probabilities")

print(f"Probabilities shape: {probabilities.shape}")
print(f"First 5 probabilities: {probabilities[:5]}")

In [ ]:
# Create submission matching the sample format
print("Creating submission file...")
# Load the sample submission to get the correct format
sample_submission = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')
submission_df = sample_submission.copy()

# Update prediction columns with our model's predictions
# TargetF1: Predicted class labels (for F1 score calculation)
# TargetRAUC: Predicted probabilities (for ROC AUC calculation)
submission_df["TargetF1"] = predictions.astype(int)
submission_df["TargetRAUC"] = probabilities

print(f"Submission shape: {submission_df.shape}")
print(f"TargetF1 distribution: {np.bincount(submission_df['TargetF1'].astype(int))}")
print(f"TargetRAUC range: {submission_df['TargetRAUC'].min():.4f} - {submission_df['TargetRAUC'].max():.4f}")

In [ ]:
# Save submission
submission_path = latest_experiment / 'submission.csv'
submission_df.to_csv(submission_path, index=False)
print(f"Submission saved to: {submission_path}")
print(f"File size: {submission_path.stat().st_size / 1024:.1f} KB")

# Add verification that cached features remain consistent across multiple calls
print("\n=== Verification: Feature Consistency Across Multiple Calls ===")
# Make predictions multiple times to verify caching works
pred1 = trainer.model.predict(X_test_for_prediction)
pred2 = trainer.model.predict(X_test_for_prediction)
pred3 = trainer.model.predict(X_test_for_prediction)

if np.array_equal(pred1, pred2) and np.array_equal(pred2, pred3):
    print("✓ Predictions are consistent across multiple calls (caching working)")
else:
    print("⚠ WARNING: Predictions differ across multiple calls!")

# Also verify probabilities if available
try:
    proba1 = trainer.model.predict_proba(X_test_for_prediction)
    proba2 = trainer.model.predict_proba(X_test_for_prediction)
    proba3 = trainer.model.predict_proba(X_test_for_prediction)
    
    if np.array_equal(proba1, proba2) and np.array_equal(proba2, proba3):
        print("✓ Probabilities are consistent across multiple calls (caching working)")
    else:
        print("⚠ WARNING: Probabilities differ across multiple calls!")
except AttributeError:
    print("Info: predict_proba not available for consistency check")

# Verify that trainer's internal cache is working (if available)
if hasattr(trainer, '_last_X') and trainer._last_X is not None:
    print(f"✓ Trainer cache is active - last X shape: {trainer._last_X.shape}")
else:
    print("Info: Trainer cache not visible or not active")

In [ ]:
# Display submission head and verification
print("\nSubmission preview:")
print(submission_df.head())

# Verify submission matches sample format
print(f"\nSubmission shape: {submission_df.shape}")
print(f"Submission columns: {list(submission_df.columns)}")
print("Submission successfully created!")

# Additional verification
print("\n--- Verification ---")
print(f"Number of rows: {len(submission_df)}")
print(f"TargetF1 unique values: {sorted(submission_df['TargetF1'].unique())}")
print(f"TargetRAUC is float: {submission_df['TargetRAUC'].dtype}")
print("All checks passed!")